In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
import pandas as pd
import os
import random

from tqdm import tqdm 
svanoo_myanimelist_dataset_path = kagglehub.dataset_download('svanoo/myanimelist-dataset')

print('Data source import complete.')


c:\Users\osher\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data source import complete.


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk(f'{svanoo_myanimelist_dataset_path}'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

C:\Users\osher\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\anime.csv
C:\Users\osher\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\anime_anime.csv
C:\Users\osher\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user.csv
C:\Users\osher\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000000.csv
C:\Users\osher\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000001.csv
C:\Users\osher\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000002.csv
C:\Users\osher\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000003.csv
C:\Users\osher\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000004.csv
C:\Users\osher\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000005.csv
C:\Users\osher\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000

In [3]:
NUM_USER_ANIME_FILES_TO_LOAD = 70
NUM_USER_ANIME_FILES_TO_LOAD = min(NUM_USER_ANIME_FILES_TO_LOAD, 70)

In [4]:
anime = pd.read_csv(f'{svanoo_myanimelist_dataset_path}/anime.csv', sep = '\t',na_values="0",
                    keep_default_na=False)


user = pd.read_csv(f'{svanoo_myanimelist_dataset_path}/user.csv', sep = '\t', keep_default_na=False)

anime_anime = pd.read_csv(f'{svanoo_myanimelist_dataset_path}/anime_anime.csv', sep = '\t', keep_default_na=False)
user_user = pd.read_csv(f'{svanoo_myanimelist_dataset_path}/user_user.csv', sep = '\t', keep_default_na=False)
list_files = ["{}/user_anime{:012d}.csv".format(svanoo_myanimelist_dataset_path,x) for x in range(NUM_USER_ANIME_FILES_TO_LOAD)]


temp_df = pd.read_csv(list_files[0], sep = '\t', keep_default_na=False)
temp_df.head(100).to_csv('temp_df.csv', index=False)

C:\Users\osher\AppData\Local\Temp\ipykernel_11368\3100813293.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  anime_anime = pd.read_csv(f'{svanoo_myanimelist_dataset_path}/anime_anime.csv', sep = '\t', keep_default_na=False)
C:\Users\osher\AppData\Local\Temp\ipykernel_11368\3100813293.py:12: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(list_files[0], sep = '\t', keep_default_na=False)


In [5]:
TITLES_TO_TAKE = 2000
anime["score_count"]= pd.to_numeric(anime["score_count"], errors='raise', downcast='float')
top_animes = anime.sort_values("score_count", ascending=False).head(TITLES_TO_TAKE)["anime_id"]
top_animes.describe()

count     2000.000000
mean     20030.066500
std      14843.454226
min          1.000000
25%       5340.000000
50%      19114.000000
75%      34128.000000
max      49926.000000
Name: anime_id, dtype: float64

In [6]:
# Set random seed for reproducibility
random.seed(42)

top_animes = set(top_animes)
output_file = "../data/filtered_user_anime.csv"
columns_to_keep = ['anime_id', 'user_id', 'score', 'favorite']

# Parameter to control what percentage of reviews to keep
sample_percentage = 1  # Keep % of reviews (adjust as needed)

if os.path.exists(output_file):
    os.remove(output_file)

first = True
total_rows_kept = 0

for file_path in tqdm(list_files, desc="Filtering user_anime files"):
    # Read in chunks to handle memory better if needed
    df = pd.read_csv(
        file_path,
        sep='\t',
        keep_default_na=False,
        na_values="-------",
        low_memory=False,
        usecols=['anime_id', 'user_id', 'score', 'favorite']  # Only read needed columns
    ).dropna()
    
    filtered = df[df["anime_id"].isin(top_animes)]
    
    if not filtered.empty:
        original_count = len(filtered)
        
        # More aggressive sampling for large datasets
        if len(filtered) > 10:  # Only sample if we have enough rows
            n_samples = max(1, int(len(filtered) * sample_percentage))
            filtered = filtered.sample(n=n_samples, random_state=42)
        
        filtered = filtered[columns_to_keep].sort_values("anime_id")
        filtered.to_csv(output_file, mode='a', index=False, header=first)
        
        total_rows_kept += len(filtered)
        print(f"File: {file_path.split('/')[-1]} - Kept {len(filtered):,} out of {original_count:,} rows")
        
        first = False

print(f"\nTotal rows kept across all files: {total_rows_kept:,}")

Filtering user_anime files:   0%|          | 0/70 [00:00<?, ?it/s]

Filtering user_anime files:   1%|▏         | 1/70 [00:06<07:53,  6.86s/it]

File: user_anime000000000000.csv - Kept 3,245,562 out of 3,245,562 rows


Filtering user_anime files:   3%|▎         | 2/70 [00:11<06:26,  5.69s/it]

File: user_anime000000000001.csv - Kept 2,220,008 out of 2,220,008 rows


...

File: user_anime000000000067.csv - Kept 2,763,416 out of 2,763,416 rows


Filtering user_anime files:  99%|█████████▊| 69/70 [06:34<00:06,  6.05s/it]

File: user_anime000000000068.csv - Kept 2,641,560 out of 2,641,560 rows


Filtering user_anime files: 100%|██████████| 70/70 [06:40<00:00,  5.72s/it]

File: user_anime000000000069.csv - Kept 2,684,253 out of 2,684,253 rows

Total rows kept across all files: 181,526,955


In [7]:
filtered.head()

,anime_id,user_id,score,favorite
2946906,1,whitealdnoah,9,0
444367,1,vonche,9,0
971632,1,waanlox,8,0
3176903,1,whyureadingthis,9,0
2535439,1,wenko,7,0


In [8]:
filtered.shape

(2684253, 4)

In [9]:
user_anime_relevant = filtered[['anime_id','user_id','score','favorite']].copy()
user_anime_relevant['score'] = pd.to_numeric(user_anime_relevant['score'],errors="coerce")
user_anime_relevant['anime_id'] = pd.to_numeric(user_anime_relevant['anime_id'],errors="coerce")
user_anime_relevant["favorite"] = pd.to_numeric(user_anime_relevant['favorite'],errors="coerce")
user_anime_relevant.head()
user_anime_relevant.dropna(inplace=True)
print(user_anime_relevant['score'].head(50))
user_anime_relevant


2946906     9.0
444367      9.0
971632      8.0
3176903     9.0
2535439     7.0
2268471    10.0
1308490     7.0
1952850     7.0
2003121     9.0
745452      8.0
1264421     5.0
3079853    10.0
1638307     9.0
1050548     3.0
3018278    10.0
2628682    10.0
1512862    10.0
3155636     7.0
1158365    10.0
2004906     5.0
1319153     9.0
2344069     9.0
1232714     7.0
103783      9.0
1344840    10.0
701530      7.0
2771329     7.0
3185503     8.0
2552771     9.0
312650      8.0
48942      10.0
1243053     9.0
2787084     9.0
2057474     8.0
1198613     8.0
390508      8.0
2848927     9.0
2000090     8.0
2290880     8.0
2283417     7.0
1824702     9.0
2440403     9.0
1834150     7.0
2723095     8.0
61395      10.0
21839       9.0
1626100    10.0
1943620    10.0
1399275    10.0
1900447    10.0
Name: score, dtype: float64


,anime_id,user_id,score,favorite
2946906,1,whitealdnoah,9.0,0
444367,1,vonche,9.0,0
971632,1,waanlox,8.0,0
3176903,1,whyureadingthis,9.0,0
2535439,1,wenko,7.0,0
...,...,...,...,...
745250,49926,vulnificus,9.0,0
290111,49926,voided232,7.0,0
3106515,49926,whoopsieee,9.0,0
2665237,49926,wesslan03,10.0,1


In [20]:
user_anime_relevant.to_csv("../data/user_anime_top2000.csv",index=False)
anime.to_csv("../data/anime.csv",index=False)
user.to_csv("../data/user.csv",index=False)
anime_anime.to_csv("../data/anime_anime.csv",index=False)
user_user.to_csv("../data/user_user.csv",index=False)